# Упражнение: BERT MLM: маскирование 80/10/10 и потери только на масках

## Задание

Подготовьте батч для предобучения BERT задачей Masked Language Modeling и посчитайте потери.
В `input.csv` — три последовательности по токенам: `seq, pos, token`. Чтобы результат был воспроизводим,
случайные числа уже «выброшены» и лежат рядом:

- `u_select` — случайное число для выбора позиции в MLM;
- `u_action` — случайное число для выбора действия;
- `random_token` — токен, который подставляется, если выпало действие «случайный токен»;
- `p_true` — вероятность, которую модель после прямого прохода дала **исходному** токену этой позиции.

Правила (как в BERT):

1. Служебные токены `[CLS]`, `[SEP]`, `[PAD]` **никогда** не выбираются, какие бы числа ни выпали.
2. Обычный токен выбирается, если `u_select < 0.15` (≈ 15% токенов).
3. Для выбранного токена: `u_action < 0.8` → заменить на `[MASK]` (действие `mask`);
   `0.8 ≤ u_action < 0.9` → заменить на `random_token` (`random`); иначе оставить как есть (`keep`).
4. Потери MLM — средняя −ln(p_true) **только по выбранным позициям** (все три действия входят в потери,
   невыбранные позиции — нет).

## Подсказки

- Сначала отфильтруйте служебные токены — среди них есть «ловушки» с маленьким `u_select`.
- Действие определяется по `u_action` только для выбранных позиций; для `keep` вход модели совпадает
  с исходным токеном, но позиция всё равно участвует в потерях.
- `mask = ~special & (u_select < 0.15)`; `np.select([u < 0.8, u < 0.9], ['mask', 'random'], 'keep')`;
  `nll = -np.log(p_true)`; итоговые потери — `nll[mask].mean()`.

## Ожидаемый формат выходного файла

CSV с колонками `seq, pos, token, input_token, action, nll`: по строке на каждую выбранную позицию
в порядке `seq`, затем `pos`; `token` — исходный токен, `input_token` — что подаётся модели
(`[MASK]`, случайный токен или исходный). Последней строкой — итог: `seq = 0, pos = 0`,
`token = [ALL]`, `input_token = [ALL]`, `action = mean`, `nll` — средние потери MLM.
`nll` округлить до 4 знаков.

## Проверка

Сохраните свой результат в `my_answer.csv` (в папке упражнения) и выполните из корня репозитория:

```bash
python tools/check_exercise.py exercises/bert_language_models
```

Эталонное решение с пояснениями — `solution.py` (загляните в него только после своей попытки).
Верное решение портал отметит сам и откроет разбор (если он запущен через `python start.py`).

## Данные

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('input.csv')
print(df.shape)
df.head()

## Решение

In [ ]:
# Ваше решение. Итог запишите в DataFrame `result` — в формате, описанном в задании.

result = None

## Проверка

In [ ]:
# Проверка: сравнение result с эталоном (запускайте после своего решения).
# Сохраняет result в my_answer.csv, подсказывает вероятную причину ошибки, верное решение отмечает в портале.
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'tools' / 'check_exercise.py').exists())
sys.path.insert(0, str(root / 'tools'))
from check_exercise import check_result

check_result(result)

---
Эталонное решение с пояснениями — в файле `solution.py`. Загляните в него после своей попытки.